In [71]:
import pandas as pd
file_path = "../data/processed/clean_book_summaries.csv"
df = pd.read_csv(file_path)
df.sample(5)

,Title,Author,Genres,Summary
11169,Hicksville,NaN,['Uncategorized'],Canadian writer Leonard Batts arrives in the ...
5830,The Werewolf of Fever Swamp,R. L. Stine,"[""Children's literature"", 'Horror', 'Speculati...",Grady Tucker and his sixteen year-old sister ...
14178,Breaking Point,Alex Flinn,['Young adult literature'],Paul has moved to Miami with his mother Laura...
435,Clouds of Witness,Dorothy L. Sayers,"['Mystery', 'Fiction', 'Suspense']","After the events of Whose Body?, Lord Peter W..."
14232,Pool of Radiance,Jim Ward,['Fantasy'],"Dragon described the novel's plot: ""Five comp..."


In [72]:
for col in ["Tone", "Pacing", "Aesthetic", "Themes"]:
    if col not in df.columns:
        df[col] = None

In [73]:
df.head()

,Title,Author,Genres,Summary,Tone,Pacing,Aesthetic,Themes
0,Animal Farm,George Orwell,"['Roman à clef', 'Satire', ""Children's literat...","Old Major, the old boar on the Manor Farm, ca...",None,None,None,None
1,A Clockwork Orange,Anthony Burgess,"['Science Fiction', 'Novella', 'Speculative fi...","Alex, a teenager living in near-future Englan...",None,None,None,None
2,The Plague,Albert Camus,"['Existentialism', 'Fiction', 'Absurdist ficti...",The text of The Plague is divided into five p...,None,None,None,None
3,An Enquiry Concerning Human Understanding,David Hume,['Uncategorized'],The argument of the Enquiry proceeds by a ser...,None,None,None,None
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",The novel posits that space around the Milky ...,None,None,None,None


In [74]:
import os
from dotenv import load_dotenv
from groq import Groq
load_dotenv("../../.env")

True

In [111]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [112]:
promptCh = """You are a STRICT semantic classification engine.

Your task is to classify a book using ONLY the predefined labels below.

You will receive:
- Title (supporting signal)
- Existing genres (noisy hints, optional)
- Book summary (PRIMARY source of truth)

CLASSIFICATION RULES:

1) The BOOK SUMMARY is the primary source. The title may help disambiguate meaning.
2) Existing genres are only hints and may be inaccurate.
3) Select labels ONLY from the allowed lists.
4) NEVER invent new labels, synonyms, or variations.
5) Labels must match EXACT spelling including underscores.
6) Be conservative. Assign labels ONLY when clearly dominant.
7) Prefer fewer strong labels over many weak ones.

OUTPUT FORMAT (STRICT):

Tone: label1,label2 OR none
Pacing: label1,label2 OR none
Aesthetic: label1,label2 OR none
Themes: label1,label2,label3 OR none

Formatting rules:
- No extra text.
- No explanations.
- No analysis.
- Use lowercase exactly as defined.
- Separate multiple labels using commas with NO spaces.

Allowed Tone Labels:
dark, melancholic, hopeful, tragic, uplifting, introspective, tense, humorous, romantic, mysterious, epic, whimsical

Allowed Pacing Labels:
slow_burn, fast_paced, character_driven, plot_driven, episodic

Allowed Aesthetic Labels:
dark_academia, cyberpunk, gothic, philosophical, cozy, surreal, dystopian, mythological, historical, high_fantasy, urban_fantasy, literary, science_fiction

Allowed Theme Labels:
identity, existentialism, power_corruption, coming_of_age, revenge, redemption, morality, survival, love, loss, technology, society, politics, war, family

INPUT:

Title:
{title}

Existing Genres (may be noisy):
{genres}

Book Summary:
{summary}
"""

In [113]:
import re
def parse_labels(llm_output):
    #Initalize empty dictionary
    data = {"Tone": "", "Pacing": "", "Aesthetic": "", "Themes": ""}
    patterns = {
        "Tone": r"Tone:\s*(.*)",
        "Pacing": r"Pacing:\s*(.*)",
        "Aesthetic": r"Aesthetic:\s*(.*)",
        "Themes": r"Themes:\s*(.*)"
    }
    for key, pattern in patterns.items():
        match = re.search(pattern, llm_output)
        if match:
            data[key] = match.group(1).strip()
    return data

In [118]:
test_chunk = df.head(50).copy()

for index, row in test_chunk.iterrows():

    user_input = f"""
Title:
{row['Title']}

Existing Genres (may be noisy):
{row['Genres']}

Book Summary:
{row['Summary']}
"""

    chat_completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0.3,
        messages=[
            {"role": "system", "content": promptCh},
            {"role": "user", "content": user_input}
        ]
    )

    llm_output = chat_completion.choices[0].message.content

    parsed_data = parse_labels(llm_output)

    for category, labels in parsed_data.items():
        test_chunk.at[index, category] = labels

    print(f"[{index+1}/50] Classified: {row['Title']}")

[1/50] Classified: Animal Farm
[2/50] Classified: A Clockwork Orange
[3/50] Classified: The Plague
[4/50] Classified: An Enquiry Concerning Human Understanding
[5/50] Classified: A Fire Upon the Deep
[6/50] Classified: All Quiet on the Western Front
[7/50] Classified: A Wizard of Earthsea
[8/50] Classified: Anyone Can Whistle
[9/50] Classified: Blade Runner 3: Replicant Night
[10/50] Classified: Blade Runner 2: The Edge of Human
[11/50] Classified: Book of Joshua
[12/50] Classified: Book of Ezra
[13/50] Classified: Book of Numbers
[14/50] Classified: Book of Ruth
[15/50] Classified: Book of Esther
[16/50] Classified: Book of Job
[17/50] Classified: Book of Hosea
[18/50] Classified: Book of Jonah
[19/50] Classified: Book of Micah
[20/50] Classified: Book of Haggai
[21/50] Classified: Crash
[22/50] Classified: Children of Dune
[23/50] Classified: Candide, ou l'Optimisme


APIConnectionError: Connection error.

In [119]:
test_chunk.head(20)

,Title,Author,Genres,Summary,Tone,Pacing,Aesthetic,Themes
0,Animal Farm,George Orwell,"['Roman à clef', 'Satire', ""Children's literat...","Old Major, the old boar on the Manor Farm, ca...","tragic, hopeful",slow_burn,none,"power_corruption, identity, morality, loss"
1,A Clockwork Orange,Anthony Burgess,"['Science Fiction', 'Novella', 'Speculative fi...","Alex, a teenager living in near-future Englan...","dark, tragic, humorous",fast_paced,dystopian,"identity, power_corruption, morality, loss, so..."
2,The Plague,Albert Camus,"['Existentialism', 'Fiction', 'Absurdist ficti...",The text of The Plague is divided into five p...,"tragic, melancholic",slow_burn,none,"existentialism, identity, morality, loss, surv..."
3,An Enquiry Concerning Human Understanding,David Hume,['Uncategorized'],The argument of the Enquiry proceeds by a ser...,"philosophical, introspective",slow_burn,literary,"existentialism, identity, morality, power_corr..."
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",The novel posits that space around the Milky ...,"hopeful, melancholic",slow_burn,science_fiction,"identity, power_corruption, redemption, surviv..."
5,All Quiet on the Western Front,Erich Maria Remarque,"['War novel', 'Roman à clef']","The book tells the story of Paul Bäumer, a Ge...","tragic, melancholic",slow_burn,none,"identity, existentialism, loss, morality, war"
6,A Wizard of Earthsea,Ursula K. Le Guin,"[""Children's literature"", 'Fantasy', 'Speculat...","Ged is a young boy on Gont, one of the larger...","introspective, hopeful, melancholic",slow_burn,high_fantasy,"identity, power_corruption, redemption, morali..."
7,Anyone Can Whistle,Arthur Laurents,['Uncategorized'],The story is set in an imaginary American tow...,"humorous, ironic","fast_paced, episodic",none,"identity, morality, redemption"
8,Blade Runner 3: Replicant Night,K. W. Jeter,"['Science Fiction', 'Speculative fiction']","Living on Mars, Deckard is acting as a consul...",none,none,science_fiction,"power_corruption, identity"
9,Blade Runner 2: The Edge of Human,K. W. Jeter,"['Science Fiction', 'Speculative fiction']",Beginning several months after the events in ...,"dark,tragic","slow_burn,plot_driven",science_fiction,"power_corruption, identity, revenge, redemptio..."


In [120]:
test_chunk['meaning'] = (
    "Summary: " + test_chunk['Summary'] +
    " | Tone: " + test_chunk['Tone'] +
    " | Pacing: " + test_chunk['Pacing'] +
    " | Aesthetic: " + test_chunk['Aesthetic'] +
    " | Themes: " + test_chunk['Themes']
)

In [121]:
test_chunk.head()

,Title,Author,Genres,Summary,Tone,Pacing,Aesthetic,Themes,meaning
0,Animal Farm,George Orwell,"['Roman à clef', 'Satire', ""Children's literat...","Old Major, the old boar on the Manor Farm, ca...","tragic, hopeful",slow_burn,none,"power_corruption, identity, morality, loss","Summary: Old Major, the old boar on the Manor..."
1,A Clockwork Orange,Anthony Burgess,"['Science Fiction', 'Novella', 'Speculative fi...","Alex, a teenager living in near-future Englan...","dark, tragic, humorous",fast_paced,dystopian,"identity, power_corruption, morality, loss, so...","Summary: Alex, a teenager living in near-futu..."
2,The Plague,Albert Camus,"['Existentialism', 'Fiction', 'Absurdist ficti...",The text of The Plague is divided into five p...,"tragic, melancholic",slow_burn,none,"existentialism, identity, morality, loss, surv...",Summary: The text of The Plague is divided in...
3,An Enquiry Concerning Human Understanding,David Hume,['Uncategorized'],The argument of the Enquiry proceeds by a ser...,"philosophical, introspective",slow_burn,literary,"existentialism, identity, morality, power_corr...",Summary: The argument of the Enquiry proceeds...
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",The novel posits that space around the Milky ...,"hopeful, melancholic",slow_burn,science_fiction,"identity, power_corruption, redemption, surviv...",Summary: The novel posits that space around t...


In [122]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 601.16it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [124]:
print("Generating embeddings .....")
embeddings = model.encode(test_chunk['meaning'].to_list(), show_progress_bar=True)

print("Embeddings shape ", embeddings.shape)

Generating embeddings .....


Batches:  50%|██████████████████████████████████████████████████████████████████▌                                                                  | 1/2 [00:00<00:00,  4.61it/s]


TypeError: 'float' object is not subscriptable

In [125]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
def search_books(query, top_n=5):
    query_vector = model.encode([query])
    similarities = cosine_similarity(query_vector, embeddings)[0]
    top_indices = np.argsort(similarities)[-top_n:][::-1]
    results = test_chunk.iloc[top_indices].copy()
    results['similarity_score'] = similarities[top_indices]
    
    return results[['Title', 'Author', 'Genres', 'similarity_score']]

In [126]:
search_books("dark academia")

,Title,Author,Genres,similarity_score
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",0.367449
20,Crash,J. G. Ballard,"['Speculative fiction', 'Fiction', 'Novel']",0.281013
43,Hamlet,William Shakespeare,['Uncategorized'],0.276242
42,Heart of Darkness,Joseph Conrad,"['Fiction', 'Novella', 'Roman à clef']",0.276242
45,Adventures of Huckleberry Finn,Mark Twain,"['Satire', ""Children's literature"", 'Fiction',...",0.276242


In [127]:
search_books("character-driven philosophical sci-fi")

,Title,Author,Genres,similarity_score
8,Blade Runner 3: Replicant Night,K. W. Jeter,"['Science Fiction', 'Speculative fiction']",0.316538
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",0.296205
20,Crash,J. G. Ballard,"['Speculative fiction', 'Fiction', 'Novel']",0.287599
27,Don Quixote,Miguel de Cervantes,"['Parody', ""Children's literature"", 'Psycholog...",0.280542
30,Darwin's Dangerous Idea,Daniel Dennett,"['Philosophy', 'Science']",0.273668


In [128]:
search_books("political dystopia.")

,Title,Author,Genres,similarity_score
34,The Metamorphosis,NaN,"[""Children's literature"", 'Absurdist fiction',...",0.217423
8,Blade Runner 3: Replicant Night,K. W. Jeter,"['Science Fiction', 'Speculative fiction']",0.216816
1,A Clockwork Orange,Anthony Burgess,"['Science Fiction', 'Novella', 'Speculative fi...",0.201063
31,Death of a Hero,Richard Aldington,['Fiction'],0.197268
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",0.195326


In [129]:
search_books("time loop story and philosophy")

,Title,Author,Genres,similarity_score
34,The Metamorphosis,NaN,"[""Children's literature"", 'Absurdist fiction',...",0.383283
7,Anyone Can Whistle,Arthur Laurents,['Uncategorized'],0.279141
33,The Trial,Franz Kafka,"['Fiction', 'Absurdist fiction', 'Novel']",0.275582
35,Fahrenheit 451,Ray Bradbury,"['Science Fiction', ""Children's literature"", '...",0.264342
31,Death of a Hero,Richard Aldington,['Fiction'],0.257882


In [130]:
search_books("moral dilemma involving artificial intelligence")

,Title,Author,Genres,similarity_score
30,Darwin's Dangerous Idea,Daniel Dennett,"['Philosophy', 'Science']",0.220223
33,The Trial,Franz Kafka,"['Fiction', 'Absurdist fiction', 'Novel']",0.193964
3,An Enquiry Concerning Human Understanding,David Hume,['Uncategorized'],0.172259
34,The Metamorphosis,NaN,"[""Children's literature"", 'Absurdist fiction',...",0.167879
4,A Fire Upon the Deep,Vernor Vinge,"['Hard science fiction', 'Science Fiction', 'S...",0.163024


In [131]:
search_books("lonely but beautiful")

,Title,Author,Genres,similarity_score
49,Leviticus,NaN,['Uncategorized'],0.220792
48,Icehenge,Kim Stanley Robinson,"['Science Fiction', 'Speculative fiction']",0.220792
47,Johnny Got His Gun,Dalton Trumbo,"['Fiction', 'Novel']",0.220792
46,Ivanhoe,Walter Scott,"['Historical fiction', 'Fiction', 'Historical ...",0.220792
45,Adventures of Huckleberry Finn,Mark Twain,"['Satire', ""Children's literature"", 'Fiction',...",0.220792
